# Baqaee & Farhi (2022) — COVID-19 Application: Equilibrium Solver

This notebook solves the disaggregated Keynesian production-network model of
Baqaee & Farhi (2022), *Supply and Demand in Disaggregated Keynesian Economies
with an Application to the COVID-19 Crisis* (AEA P&P), using the data layer
from `01_data_layer.ipynb`.

**Model** (`src/model.jl`): a square system of nonlinear equations (2D = 668
unknowns: prices `p` and inverse demands `λ`) ported from the AMPL file
`Standard_form_Covid_HtM3_logCES.mod`. The sticky-labor complementarity is
replaced by the smooth Fischer–Burmeister function. Solved with NLsolve
(trust region, finite-difference Jacobian).

**Verified results:** the no-shock equilibrium (t=0) reproduces the trivial
fixed point p = 1, λ = init_lambda exactly, and a small COVID shock produces
a Keynesian demand-constrained equilibrium with real GDP decline.


In [2]:
# Setup: same robust project-root resolution as the data-layer notebook
cd(dirname(Base.active_project()))
using Printf, LinearAlgebra, Statistics
include("src/io_table.jl")
include("src/shocks.jl")
include("src/network.jl")
include("src/model.jl")

DATA_DIR = "data"
N, YEAR = 66, 2015

println("Modules loaded — project root: $(pwd())");


Modules loaded — project root: /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication2


In [3]:
io = load_io_table(joinpath(DATA_DIR, "IO_data_2018.mat"); N=N, year=YEAR)
shocks = load_shocks(DATA_DIR; N=N)
sf = build_standard_form(io)
D = sf.D
@printf("D = %d (5×%d+4), Domar sum = %.4f\n", D, N, sum(sf.Domar))


Loading IO table from data/IO_data_2018.mat ... done. N=66, year=2015, beta sum=1.000000
Loading shocks from data ...
  BLS shock: min=-0.5409, max=0.0206
  PCE shock: min=-0.9182, max=0.2089
Building standard form Ω_re: N=66, D=334
  Psi_re[1,2:N+1] sum = 1.807659
D = 334 (5×66+4), Domar sum = 1.8077


---
## 1. Model Setup

`make_model` builds the base `MCPModel` replicating the MATLAB driver's
calibration (`Master_file_3.m` lines 328–335) **exactly**:

- `init_lambda[1] = 1` (consumption today) — *not* the raw Domar weight
- `init_lambda[D-2] = 0` (HtM consumer; chi ≡ 0)
- `init_lambda[D-1] = 2` (Ricardian consumer)
- `init_lambda[D] = 1` (tomorrow's consumption, numeraire)
- `phi_htm[labor] = 1 - htm_share`; benchmark uses `htm_share = 0` (full insurance)

**Why this matters:** the raw first row of `Psi_re` is *not* the AMPL initial
point. Using it directly makes the solver wander to a spurious solution
(prices deviating from 1 at t=0).


In [4]:
# Benchmark regime (Fig 2): sigma=1, epsilon=0.6, eta=0.6, theta1=0.2
m0 = make_model(sf; htm_share=0.0, benchmark=true,
                sigma=1.0, epsilon=0.6, eta=0.6, theta1=0.2)

@printf("init_lambda: λ[1]=%.4f  λ[HtM]=%.4f  λ[Ric]=%.4f  λ[tomorrow]=%.4f\n",
        m0.init_lambda[1], m0.init_lambda[D-2], m0.init_lambda[D-1], m0.init_lambda[D])
@printf("phi_htm[labor] = %.2f (benchmark: full insurance)\n", m0.phi_htm[3*N+2])


init_lambda: λ[1]=1.0000  λ[HtM]=0.0000  λ[Ric]=2.0000  λ[tomorrow]=1.0000
phi_htm[labor] = 1.00 (benchmark: full insurance)


---
## 2. Residual at the Initial Point

If the calibration is faithful, the MATLAB-adjusted initial point
`z0 = (p, λ) = (1, init_lambda)` is an **exact equilibrium** of the no-shock
economy: the residual ‖F(z0)‖ should be ≈ 0.


In [5]:
z0 = vcat(m0.init_p, m0.init_lambda)
F_test = zeros(2*D)
equilibrium_residual!(F_test, z0, m0)
resid0 = norm(F_test)
@printf("Residual norm at initial point: %.2e\n", resid0)
if resid0 < 1e-9
    println("✅ Initial point is an equilibrium — calibration is MATLAB-faithful.")
else
    println("⚠️  Residual = $(resid0) — check calibration:")
    idx = sortperm(abs.(F_test), rev=true)[1:5]
    for i in idx
        i ≤ D ? @printf("  F[%3d] (p)     = %.4e\n", i, F_test[i]) :
                 @printf("  F[%3d] (λ)     = %.4e\n", i-D, F_test[i])
    end
end


Residual norm at initial point: 1.00e-10
✅ Initial point is an equilibrium — calibration is MATLAB-faithful.


---
## 3. Solve the No-Shock Equilibrium (t = 0)

At t=0 there is no shock: A = 1, B = 1. The equilibrium must be the trivial
one — all prices 1, all λ at their initial values — and the solver should
converge in a handful of iterations from the initial point.


In [6]:
p0, λ0, conv0, iters0 = solve_equilibrium(m0, z0=z0, tol=1e-10, maxiter=200)
@printf("Converged: %s, iterations: %d\n", conv0 ? "yes" : "no", iters0)
@printf("Max |p − 1| = %.2e   (expect ≈ 0)\n", maximum(abs.(p0 .- 1.0)))
@printf("p[%d] (numeraire) = %.8f\n", D, p0[D])
@printf("λ[1] (nominal GDP) = %.6f   (expect 1.0)\n", λ0[1])

# Verification gates
ok_prices = maximum(abs.(p0 .- 1.0)) < 1e-6
ok_numer  = abs(p0[D] - 1.0) < 1e-8
ok_lambda = abs(λ0[1] - 1.0) < 1e-6
println(ok_prices && ok_numer && ok_lambda ?
        "✅ t=0 equilibrium is trivial: p=1, λ=init_lambda." :
        "⚠️  t=0 equilibrium deviates from trivial — investigate.")


Converged: yes, iterations: 0
Max |p − 1| = 0.00e+00   (expect ≈ 0)
p[334] (numeraire) = 1.00000000
λ[1] (nominal GDP) = 1.000000   (expect 1.0)
✅ t=0 equilibrium is trivial: p=1, λ=init_lambda.


---
## 4. COVID-19 Shock (Baseline, t = 0.05)

Apply shock_type = 1 (supply + sectoral demand + aggregate demand), exactly
as `Master_file_3.m` does:

- **Supply:** `A[labor] = 1 + t·BLS_shock` — the BLS hours change enters with
  a **plus** sign because MATLAB first flips it (`shock = -BLS_shock`).
  A hard-hit sector (BLS = −0.54) gets A = 1 − 0.54t < 1 → supply contraction.
- **Sectoral demand:** `B[1, 2:N+1] = (1−0.66t) + 0.66t·(1+PCE)`, then the row
  is renormalized so `Ω[1,:]·B[1,:]′ = 1`.
- **Aggregate demand:** `B[5N+3, D] = 1 + 0.105` (counter ≥ 2), renormalized
  the same way.

The numeraire row `λ[334]` (tomorrow's consumption) is pinned by factor
clearing (`keynes[334]=0`), which prevents the degenerate self-reference that
otherwise collapses GDP.


In [7]:
# Build the t=0.05 shocked model manually (mirrors eq_continuation internals)
t = 0.05
A_s = ones(D)
A_s[(3*N+2):(4*N+1)] .= 1.0 .+ t .* shocks.BLS_shock

B_s = ones(D, D)
B_s[1, 2:(N+1)] .= (1 - t*0.66) .+ t*0.66 .* (1.0 .+ shocks.PCE_shock)
B_s[1, :] ./= sum(sf.Omega_re[1, :] .* B_s[1, :])
B_s[5*N+3, D] = 1.0 + 0.105
B_s[5*N+3, :] ./= sum(sf.Omega_re[5*N+3, :] .* B_s[5*N+3, :])

m_t = MCPModel(D, N, sf.Omega_re, sf.factor, sf.keynes,
               m0.theta, m0.cobb_douglas, sf.phi, m0.phi_htm,
               ones(D), ones(D), A_s, B_s,
               m0.init_lambda, m0.init_p, sf.chi)

p_t, λ_t, conv_t, iters_t = solve_equilibrium(m_t, z0=vcat(p0, λ0), tol=1e-10, maxiter=500)
@printf("t=0.05: converged=%s, iters=%d\n", conv_t ? "yes" : "no", iters_t)
@printf("Real GDP  λ[1]/p[1] = %.4f   (t=0: 1.0)\n", λ_t[1]/p_t[1])
@printf("Nominal GDP λ[1]   = %.4f\n", λ_t[1])
@printf("Inflation p[1]     = %.4f\n", p_t[1])
@printf("λ[334] (tomorrow)  = %.4f   (pinned by factor clearing)\n", λ_t[D])

# Keynesian unemployment: labor sectors with λ/p < A·λ̄ (demand-constrained)
lab = (3*N+2):(4*N+1)
unemp = count(λ_t[lab] ./ p_t[lab] .< A_s[lab] .* m0.init_lambda[lab] .- 1e-6)
@printf("Demand-constrained (unemployed) labor sectors: %d / %d\n", unemp, N)


t=0.05: converged=yes, iters=6
Real GDP  λ[1]/p[1] = 0.9569   (t=0: 1.0)
Nominal GDP λ[1]   = 0.9050
Inflation p[1]     = 0.9457
λ[334] (tomorrow)  = 1.0000   (pinned by factor clearing)
Demand-constrained (unemployed) labor sectors: 66 / 66


---
## 5. Continuation: Shock Grid

`eq_continuation` solves the equilibrium for each `t` in a grid, using the
previous solution as initializer (path-following / homotopy). This mirrors
the outer loop of `Master_file_3.m` (`shock_grid`), which for the benchmark
runs `[0 (0.1:.05:.9) .95 .98 .99 .995 1]`.

> ⚠️ Memory note: each solve needs the FD Jacobian of the 668-variable system.
> The full grid (~25 points) fits comfortably on the host Mac (32 GB) but can
> OOM a 5 GB container. Start with a small grid here.


In [8]:
t_grid = [0.0, 0.01, 0.05, 0.10]
res = eq_continuation(m0, shocks.BLS_shock, shocks.PCE_shock, t_grid, show_trace=true)

@printf("\n%-8s %-10s %-10s %-10s %-8s\n", "t", "RGDP", "Hulten", "nomGDP", "retcode")
for ti in 1:length(t_grid)
    @printf("%-8.3f %-10.6f %-10.6f %-10.6f %-8d\n",
            t_grid[ti], res["GDP"][ti], res["Hulten"][ti],
            res["nominal_GDP"][ti], res["retcodes"][ti])
end

if all(res["retcodes"] .== 0)
    println("✅ All solves converged.")
else
    println("⚠️  Some solves failed — GDP numbers are unreliable for those t.")
end


t=0.000  GDP=1.000000  nomGDP=1.000000  conv=1
t=0.010  GDP=0.957185  nomGDP=0.904977  conv=1
t=0.050  GDP=0.956937  nomGDP=0.904977  conv=1
t=0.100  GDP=0.956614  nomGDP=0.904977  conv=1

t        RGDP       Hulten     nomGDP     retcode 
0.000    1.000000   1.000000   1.000000   0       
0.010    0.957185   0.999414   0.904977   0       
0.050    0.956937   0.997061   0.904977   0       
0.100    0.956614   0.994099   0.904977   0       
✅ All solves converged.


---
## 6. CSV Export

Save the results of this notebook as CSV files so they can be consumed
outside the Julia environment (e.g. for analysis or plotting on the host).
Files are written to `data/results/notebook02/`.


In [9]:
using CSV, DataFrames

# Create output directory
outdir = joinpath("data", "results", "notebook02")
mkpath(outdir)

# Write the continuation results (from section 5, stored in `res`)
if @isdefined(res) && res isa Dict
    df = DataFrame(
        t = res["t_grid"],
        RGDP = res["GDP"],
        nominal_GDP = res["nominal_GDP"],
        Hulten = res["Hulten"],
        p1 = res["prices"][1, :],
        retcode = res["retcodes"],
    )
    CSV.write(joinpath(outdir, "continuation.csv"), df)
    println("✅ Written continuation.csv")
else
    println("⚠️  No `res` variable from section 5 — run the continuation cell first.")
end

# Write the manual shock test (from section 4, stored in m_t, p_t, λ_t)
if @isdefined(p_t) && @isdefined(λ_t)
    df2 = DataFrame(
        variable = ["Real GDP", "Nominal GDP", "p[1] (inflation)", "λ[334] (numeraire)"],
        value = [λ_t[1]/p_t[1], λ_t[1], p_t[1], λ_t[334]],
    )
    CSV.write(joinpath(outdir, "manual_shock_t005.csv"), df2)
    println("✅ Written manual_shock_t005.csv")
end

# Write the model parameters for reference
if @isdefined(m0)
    df3 = DataFrame(
        param = ["D", "N", "htm_share", "benchmark", "sigma", "epsilon", "eta", "theta1"],
        value = [m0.D, m0.N, 0.0, "true", 1.0, 0.6, 0.6, 0.2],
    )
    CSV.write(joinpath(outdir, "parameters.csv"), df3)
    println("✅ Written parameters.csv")
end

println("✅ All CSV files written to $(joinpath(pwd(), outdir))")


✅ Written continuation.csv
✅ Written manual_shock_t005.csv
✅ Written parameters.csv
✅ All CSV files written to /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication2/data/results/notebook02


---
## 7. Verification Summary

| Check | Result |
|-------|--------|
| Residual at MATLAB-adjusted init point ≈ 0 | ✅ |
| t=0 equilibrium trivial (p=1, λ=init_lambda) | ✅ |
| λ[334] pinned by factor clearing (no collapse) | ✅ |
| COVID shock produces demand-constrained sectors | ✅ |
| Continuation converges on small grid | ✅ |
| CSV output saved | ✅ |
| GDP declines monotonically in t | check output above |

**Next steps:**
- Run the full benchmark grid on the host Mac: `shock_grid = [0 (0.1:.05:.9) .95 .98 .99 .995 1]`
- Reproduce Figure 2 (benchmark bars), Figure 3 (Cobb–Douglas), Figure 4 (HtM sweep)
- Validate against the paper's reported numbers (`VALIDATION.md`)


In [ ]:
println("✅ Notebook complete — equilibrium solver verified.")
